# Goal is to clean the raw ultrachat dataset ad prepare it to be clean Qwen-ready converstations

## 🚀 STEP 1 - n3ml Fix ll messages format

### - 1st problem we noticed was that (Messages) are "String" they're not [Lists]
### - 2nd problem there're missing commas which give us invalid literal_eval() errors

In [1]:
import ast
import pandas as pd
import json 
from transformers import AutoTokenizer 
from sklearn.model_selection import train_test_split


In [2]:
df = pd.read_csv(r"C:\Users\user\Desktop\Lebanese Assistant\lebanese-assistant\datasets\raw\ultrachats\train_sft.csv",
    engine="python",
    on_bad_lines="skip"
)

df.head()

,prompt,prompt_id,messages
0,These instructions apply to section-based them...,f0e37e9f7800261167ce91143f98f511f768847236f133...,"[{'content': ""These instructions apply to sect..."
1,Which famous landmarks should I visit in Londo...,f5025bdcae61bb77fd98a4d6cd6ba8e0199a098cfebcf6...,[{'content': 'Which famous landmarks should I ...
2,Write a comprehensive blog post of at least 10...,6db663a4d2671b41e0038c43c39f79cf909b10987dc595...,"[{'content': ""Write a comprehensive blog post ..."
3,"De León, previewing the speech he will give to...",dd1afba7d2151b0695edea838378c8fd086d538e62a664...,"[{'content': 'De León, previewing the speech h..."
4,Write an essay that evaluates the positive and...,cbf683405d8fe0221a42560cec50307d5fa9efa160c49d...,[{'content': 'Write an essay that evaluates th...


In [3]:
df = df[['messages']]
df.head()

,messages
0,"[{'content': ""These instructions apply to sect..."
1,[{'content': 'Which famous landmarks should I ...
2,"[{'content': ""Write a comprehensive blog post ..."
3,"[{'content': 'De León, previewing the speech h..."
4,[{'content': 'Write an essay that evaluates th...


### Fix Parsing 

In [4]:
# safe parsing function hta n3ml handling ll different formats 
def safe_parse(x):
    if pd.isna(x):
        return None

    try:
        return ast.literal_eval(x)
    except:
        try:
            # fix broken UltraChat formatting
            x = x.replace("}\n {", "},\n {")
            return ast.literal_eval(x)
        except:
            return None

In [5]:
# Apply the safe parsing to the messages column 
df["parsed_messages"] = df["messages"].apply(safe_parse) # zdna column jdd w 3bayne b eemt messgs parsed 
print(len(df))

207866


### Must remove broken rows 

In [6]:
df = df[df["parsed_messages"].notnull()].reset_index(drop=True) #bnsheel l rows mn l parsed msgs yle msh true w bnhton bl dataset

In [7]:
print(type(df["parsed_messages"].iloc[0]))
print(df["parsed_messages"].iloc[0][:2])

<class 'list'>
[{'content': "These instructions apply to section-based themes (Responsive 6.0+, Retina 4.0+, Parallax 3.0+ Turbo 2.0+, Mobilia 5.0+). What theme version am I using?\nOn your Collections pages & Featured Collections sections, you can easily show the secondary image of a product on hover by enabling one of the theme's built-in settings!\nYour Collection pages & Featured Collections sections will now display the secondary product image just by hovering over that product image thumbnail.\nDoes this feature apply to all sections of the theme or just specific ones as listed in the text material?", 'role': 'user'}, {'content': 'This feature only applies to Collection pages and Featured Collections sections of the section-based themes listed in the text material.', 'role': 'assistant'}]


In [8]:
def clean_conversation(messages):
    if not isinstance(messages, list):
        return []

    cleaned = []

    for m in messages:
        if isinstance(m, dict):
            role = m.get("role", "")
            content = m.get("content", "")
            cleaned.append(f"{role}: {content}")

    return cleaned

In [9]:
df["parsed_messages"].apply(type).value_counts()


parsed_messages
<class 'list'>    207863
Name: count, dtype: int64

In [10]:
df[df["parsed_messages"].isna()]

,messages,parsed_messages


In [11]:
df["parsed_messages"] = df["parsed_messages"].apply(
    lambda x: x if isinstance(x, list) else []
)

In [12]:
df["clean_messages"] = df["parsed_messages"].apply(clean_conversation)

In [13]:
df = df[df["clean_messages"].apply(len) > 0]

In [14]:
def to_qwen(messages):
    return {
        "messages": messages
    }

In [15]:
df["qwen"] = df["clean_messages"].apply(to_qwen)

In [16]:
output_path = "../datasets/processed/qwen_dataset.jsonl"

with open(output_path, "w", encoding="utf-8") as f:
    for item in df["qwen"]:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

In [17]:
tokenizer = AutoTokenizer.from_pretrained(
    "Qwen/Qwen3-0.6B" ,  trust_remote_code=True
)

In [18]:
df

,messages,parsed_messages,clean_messages,qwen
0,"[{'content': ""These instructions apply to sect...",[{'content': 'These instructions apply to sect...,[user: These instructions apply to section-bas...,{'messages': ['user: These instructions apply ...
1,[{'content': 'Which famous landmarks should I ...,[{'content': 'Which famous landmarks should I ...,[user: Which famous landmarks should I visit i...,{'messages': ['user: Which famous landmarks sh...
2,"[{'content': ""Write a comprehensive blog post ...",[{'content': 'Write a comprehensive blog post ...,[user: Write a comprehensive blog post of at l...,{'messages': ['user: Write a comprehensive blo...
3,"[{'content': 'De León, previewing the speech h...","[{'content': 'De León, previewing the speech h...","[user: De León, previewing the speech he will ...","{'messages': ['user: De León, previewing the s..."
4,[{'content': 'Write an essay that evaluates th...,[{'content': 'Write an essay that evaluates th...,[user: Write an essay that evaluates the posit...,{'messages': ['user: Write an essay that evalu...
...,...,...,...,...
207858,"[{'content': ""Write a 5-7 page short story in ...",[{'content': 'Write a 5-7 page short story in ...,[user: Write a 5-7 page short story in third p...,{'messages': ['user: Write a 5-7 page short st...
207859,"[{'content': ""Write a natural-sounding dialogu...",[{'content': 'Write a natural-sounding dialogu...,[user: Write a natural-sounding dialogue betwe...,{'messages': ['user: Write a natural-sounding ...
207860,[{'content': 'Write instructions on how to mak...,[{'content': 'Write instructions on how to mak...,[user: Write instructions on how to make a del...,{'messages': ['user: Write instructions on how...
207861,[{'content': 'Write a heartfelt letter to a fa...,[{'content': 'Write a heartfelt letter to a fa...,[user: Write a heartfelt letter to a family me...,{'messages': ['user: Write a heartfelt letter ...


In [19]:
df.columns


Index(['messages', 'parsed_messages', 'clean_messages', 'qwen'], dtype='object')

In [20]:
df.head(1)

,messages,parsed_messages,clean_messages,qwen
0,"[{'content': ""These instructions apply to sect...",[{'content': 'These instructions apply to sect...,[user: These instructions apply to section-bas...,{'messages': ['user: These instructions apply ...


In [21]:
df["qwen"].iloc[0]

{'messages': ["user: These instructions apply to section-based themes (Responsive 6.0+, Retina 4.0+, Parallax 3.0+ Turbo 2.0+, Mobilia 5.0+). What theme version am I using?\nOn your Collections pages & Featured Collections sections, you can easily show the secondary image of a product on hover by enabling one of the theme's built-in settings!\nYour Collection pages & Featured Collections sections will now display the secondary product image just by hovering over that product image thumbnail.\nDoes this feature apply to all sections of the theme or just specific ones as listed in the text material?",
  'assistant: This feature only applies to Collection pages and Featured Collections sections of the section-based themes listed in the text material.',
  'user: Can you guide me through the process of enabling the secondary image hover feature on my Collection pages and Featured Collections sections?',
  "assistant: Sure, here are the steps to enable the secondary image hover feature on yo

In [22]:
type(df["qwen"].iloc[0])

dict

In [23]:
type(df["qwen"].iloc[0]["messages"])

list

In [24]:
def convert_to_qwen(messages):
    conversation = []

    for msg in messages:
        if msg.startswith("user: "):
            conversation.append({
                "role": "user",
                "content": msg.replace("user: ", "", 1)
            })

        elif msg.startswith("assistant: "):
            conversation.append({
                "role": "assistant",
                "content": msg.replace("assistant: ", "", 1)
            })

    return conversation

In [25]:
df["qwen_messages"] = df["qwen"].apply(
    lambda x: convert_to_qwen(x["messages"])
)

In [26]:
df["qwen_messages"].iloc[0]

[{'role': 'user',
  'content': "These instructions apply to section-based themes (Responsive 6.0+, Retina 4.0+, Parallax 3.0+ Turbo 2.0+, Mobilia 5.0+). What theme version am I using?\nOn your Collections pages & Featured Collections sections, you can easily show the secondary image of a product on hover by enabling one of the theme's built-in settings!\nYour Collection pages & Featured Collections sections will now display the secondary product image just by hovering over that product image thumbnail.\nDoes this feature apply to all sections of the theme or just specific ones as listed in the text material?"},
 {'role': 'assistant',
  'content': 'This feature only applies to Collection pages and Featured Collections sections of the section-based themes listed in the text material.'},
 {'role': 'user',
  'content': 'Can you guide me through the process of enabling the secondary image hover feature on my Collection pages and Featured Collections sections?'},
 {'role': 'assistant',
  'co

In [27]:
sample = df["qwen_messages"].iloc[0]

formatted_text = tokenizer.apply_chat_template(
    sample,
    tokenize=False,
    add_generation_prompt=False
)

print(formatted_text[:2000])

<|im_start|>user
These instructions apply to section-based themes (Responsive 6.0+, Retina 4.0+, Parallax 3.0+ Turbo 2.0+, Mobilia 5.0+). What theme version am I using?
On your Collections pages & Featured Collections sections, you can easily show the secondary image of a product on hover by enabling one of the theme's built-in settings!
Your Collection pages & Featured Collections sections will now display the secondary product image just by hovering over that product image thumbnail.
Does this feature apply to all sections of the theme or just specific ones as listed in the text material?<|im_end|>
<|im_start|>assistant
This feature only applies to Collection pages and Featured Collections sections of the section-based themes listed in the text material.<|im_end|>
<|im_start|>user
Can you guide me through the process of enabling the secondary image hover feature on my Collection pages and Featured Collections sections?<|im_end|>
<|im_start|>assistant
Sure, here are the steps to enabl

In [28]:
def format_chat(messages):
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

df["text"] = df["qwen_messages"].apply(format_chat)

In [29]:
print(df["text"].iloc[0][:500])

<|im_start|>user
These instructions apply to section-based themes (Responsive 6.0+, Retina 4.0+, Parallax 3.0+ Turbo 2.0+, Mobilia 5.0+). What theme version am I using?
On your Collections pages & Featured Collections sections, you can easily show the secondary image of a product on hover by enabling one of the theme's built-in settings!
Your Collection pages & Featured Collections sections will now display the secondary product image just by hovering over that product image thumbnail.
Does this


In [30]:
sample_df = df.sample(
    n=10000,
    random_state=42
)

In [31]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(
    sample_df,
    test_size=0.1,
    random_state=42
)

In [32]:
train_save = train_df[["text"]]
val_save = val_df[["text"]]

In [33]:
import json

train_save.to_json(
    r"C:\Users\user\Desktop\Lebanese Assistant\lebanese-assistant\datasets\processed/train.jsonl",
    orient="records",
    lines=True,
    force_ascii=False
)

val_save.to_json(
    r"C:\Users\user\Desktop\Lebanese Assistant\lebanese-assistant\datasets\processed/val.jsonl",
    orient="records",
    lines=True,
    force_ascii=False
)

In [34]:
import os

os.listdir("../datasets/processed")

['qwen_dataset.jsonl', 'train.jsonl', 'val.jsonl']

In [36]:
pd.read_json("../datasets/processed/train.jsonl", lines=True)

,text
0,<|im_start|>user\nWrite instructions for makin...
1,<|im_start|>user\nGiven the text: Rotherham Un...
2,<|im_start|>user\nWhat are some unique outdoor...
3,<|im_start|>user\nCould you continue the story...
4,<|im_start|>user\nHow has Mozambique's cultura...
...,...
8995,<|im_start|>user\nCan you summarize the recent...
8996,<|im_start|>user\nCreate a comprehensive and i...
8997,<|im_start|>user\nCan you describe in detail t...
8998,<|im_start|>user\nWhat measures do African nat...


In [38]:
import pandas as pd

val_df = pd.read_json(
    r"C:\Users\user\Desktop\Lebanese Assistant\lebanese-assistant\datasets\processed\val.jsonl",
    lines=True
)

print(val_df.shape)

(1000, 1)


In [40]:
import pandas as pd

train_df = pd.read_json(
     r"C:\Users\user\Desktop\Lebanese Assistant\lebanese-assistant\datasets\processed\train.jsonl",
    lines=True
)

print(train_df.columns)
print(train_df.head(1))

Index(['text'], dtype='object')
                                                text
0  <|im_start|>user\nWrite instructions for makin...


In [7]:
import json
import os

combined_file = r"C:\Users\user\Desktop\Lebanese Assistant\lebanese-assistant\datasets\processed\finalDataset.jsonl"
print(f"--- 🔍 فحص ملف الداتا: {combined_file} ---")

try:
    with open(combined_file, "r", encoding="utf-8") as f:
        for i in range(2):
            line = f.readline()
            if not line:
                break
            data = json.loads(line)
            print(f"\n🔹 [السطر {i+1}] - نوع البيانات (Type): {type(data)}")
            if isinstance(data, dict):
                print(f"   المفاتيح الموجودة (Keys): {list(data.keys())}")
                
    with open(combined_file, "r", encoding="utf-8") as f:
        total_lines = sum(1 for line in f if line.strip())
    print(f"\n📊 إجمالي عدد الأسطر (Shape) في الملف هو: {total_lines} أسطر.")
except FileNotFoundError:
    print(f"❌ خطأ: ملف {combined_file} مش موجود! تأكد من المسار.")


--- 🔍 فحص ملف الداتا: C:\Users\user\Desktop\Lebanese Assistant\lebanese-assistant\datasets\processed\finalDataset.jsonl ---

🔹 [السطر 1] - نوع البيانات (Type): <class 'dict'>
   المفاتيح الموجودة (Keys): ['text']

🔹 [السطر 2] - نوع البيانات (Type): <class 'dict'>
   المفاتيح الموجودة (Keys): ['messages']

📊 إجمالي عدد الأسطر (Shape) في الملف هو: 15000 أسطر.


In [8]:
import json
import re

combined_file = r"C:\Users\user\Desktop\Lebanese Assistant\lebanese-assistant\datasets\processed\finalDataset.jsonl"

system_prompt = {
    "role": "system", 
    "content": "أنت مساعد ذكي لبناني. بتفهم وبتتكلم بالعامية اللبنانية وباللغة الإنجليزية بطلاقة، وأسلوبك دايماً قريب للقلب ومساعد."
}

all_clean_data = []
processed_count = 0
skipped_count = 0

print("⏳ جاري تفكيك النص واستخراج الـ 15,000 سطر بالكامل...")

with open(combined_file, "r", encoding="utf-8") as f:
    for idx, line in enumerate(f):
        if not line.strip(): continue
        try:
            data = json.loads(line)
            clean_messages_list = []
            
            # الشكل الأول: حقل 'messages' جاهز (الداتا اللبنانية)
            if isinstance(data, dict) and "messages" in data:
                raw_messages = data["messages"]
                for msg in raw_messages:
                    if isinstance(msg, dict):
                        clean_messages_list.append(msg)
                    elif isinstance(msg, str):
                        if msg.startswith("user:"):
                            clean_messages_list.append({"role": "user", "content": msg.replace("user:", "", 1).strip()})
                        elif msg.startswith("assistant:"):
                            clean_messages_list.append({"role": "assistant", "content": msg.replace("assistant:", "", 1).strip()})

            # الشكل الثاني: حقل 'text' وبداخله تاغات ChatML (داتا UltraChat المفرمتة)
            elif isinstance(data, dict) and "text" in data:
                text_content = data["text"]
                pattern = r"<\|im_start\|>(user|assistant|system)\n(.*?)(?=<\|im_end\|>)"
                matches = re.findall(pattern, text_content, re.DOTALL)
                for role, content in matches:
                    clean_messages_list.append({"role": role, "content": content.strip()})
            else:
                skipped_count += 1
                continue
                
            if not clean_messages_list:
                skipped_count += 1
                continue
                
            if clean_messages_list[0].get("role") == "system":
                clean_messages_list = clean_messages_list[1:]
            
            clean_row = {"messages": [system_prompt] + clean_messages_list}
            all_clean_data.append(clean_row)
            processed_count += 1
        except json.JSONDecodeError:
            skipped_count += 1
            continue

print(f"\n✨ انتهت المعالجة الشاملة بنجاح ساحق!")
print(f"✅ إجمالي الأسطر المستخرجة والمظبطة: {processed_count}")
print(f"⚠️ أسطر خربانة أو فارغة وتم تخطيها: {skipped_count}")


⏳ جاري تفكيك النص واستخراج الـ 15,000 سطر بالكامل...

✨ انتهت المعالجة الشاملة بنجاح ساحق!
✅ إجمالي الأسطر المستخرجة والمظبطة: 15000
⚠️ أسطر خربانة أو فارغة وتم تخطيها: 0


In [9]:
import random

print("🎲 جاري خلط البيانات عشوائياً (Shuffling)...")
random.shuffle(all_clean_data)
print("✅ تم خلط البيانات بنجاح!")


🎲 جاري خلط البيانات عشوائياً (Shuffling)...
✅ تم خلط البيانات بنجاح!


In [11]:
import json
import os

split_ratio = 0.9
split_index = int(len(all_clean_data) * split_ratio)

train_data = all_clean_data[:split_index]
eval_data = all_clean_data[split_index:]

output_dir = r"C:\Users\user\Desktop\Lebanese Assistant\lebanese-assistant\datasets\processed"
train_path = os.path.join(output_dir, "train.jsonl")
eval_path = os.path.join(output_dir, "eval.jsonl")

print("💾 جاري حفظ ملفات الـ JSONL الجديدة الحقيقية...")
with open(train_path, "w", encoding="utf-8") as f:
    for row in train_data:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

with open(eval_path, "w", encoding="utf-8") as f:
    for row in eval_data:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

print(f"\n🎉 انتهت المرحلة الأولى! الملفات انحفظت بـ: {output_dir}")
print(f"📝 ملف التدريب (train.jsonl): {len(train_data)} سطر.")
print(f"📊 ملف التقييم (eval.jsonl): {len(eval_data)} سطر.")


💾 جاري حفظ ملفات الـ JSONL الجديدة الحقيقية...

🎉 انتهت المرحلة الأولى! الملفات انحفظت بـ: C:\Users\user\Desktop\Lebanese Assistant\lebanese-assistant\datasets\processed
📝 ملف التدريب (train.jsonl): 13500 سطر.
📊 ملف التقييم (eval.jsonl): 1500 سطر.


In [14]:
import json
import os

# مسارات الملفات التي تم إنشاؤها
output_dir = r"C:\Users\user\Desktop\Lebanese Assistant\lebanese-assistant\datasets\processed"
train_path = os.path.join(output_dir, "train.jsonl")
eval_path = os.path.join(output_dir, "eval.jsonl")

def validate_qwen_dataset(file_path):
    print(f"\n--- ⏳ جاري الفحص الدقيق لملف: {os.path.basename(file_path)} ---")
    
    if not os.path.exists(file_path):
        print(f"❌ خطأ: الملف غير موجود على المسار المحدد!")
        return False
        
    errors = 0
    null_count = 0
    wrong_format_count = 0
    total_rows = 0
    
    with open(file_path, "r", encoding="utf-8") as f:
        for idx, line in enumerate(f):
            if not line.strip():
                continue
            total_rows += 1
            
            try:
                data = json.loads(line)
                
                # 1. فحص الهيكل الأساسي (يجب أن يحتوي على messages وتكون لستة)
                if not isinstance(data, dict) or "messages" not in data or not isinstance(data["messages"], list):
                    print(f"🚨 [السطر {idx+1}]: خطأ في الهيكل الأساسي! (يجب أن يكون Dict يحتوي على لستة messages)")
                    wrong_format_count += 1
                    errors += 1
                    continue
                
                messages = data["messages"]
                
                # 2. فحص محتوى الرسائل بالداخل
                for msg_idx, msg in enumerate(messages):
                    # فحص إذا كانت الرسالة null أو ليست dict
                    if msg is None or not isinstance(msg, dict):
                        print(f"🚨 [السطر {idx+1}][الرسالة {msg_idx}]: الرسالة فارغة null أو ليست كائن!")
                        null_count += 1
                        errors += 1
                        continue
                    
                    # فحص وجود المفاتيح الأساسية role و content
                    if "role" not in msg or "content" not in msg:
                        print(f"🚨 [السطر {idx+1}][الرسالة {msg_idx}]: حقول ناقصة! يجب وجود 'role' و 'content'")
                        wrong_format_count += 1
                        errors += 1
                        continue
                        
                    # فحص القيم إذا كانت فارغة أو null
                    role_val = msg["role"]
                    content_val = msg["content"]
                    
                    if role_val is None or content_val is None:
                        print(f"🚨 [السطر {idx+1}][الرسالة {msg_idx}]: قيمة حقل role أو content تحتوي على null!")
                        null_count += 1
                        errors += 1
                        continue
                        
                    # فحص محتوى النص إذا كان مجرد فراغات وايت سبيس
                    if isinstance(content_val, str) and not content_val.strip():
                        print(f"🚨 [السطر {idx+1}][الرسالة {msg_idx}]: الـ content عبارة عن نص فارغ (Spaces/Empty)!")
                        null_count += 1
                        errors += 1
                        
                    # تأكيد الأدوار الصحيحة لـ Qwen 3.5
                    if role_val not in ["system", "user", "assistant"]:
                        print(f"🚨 [السطر {idx+1}][الرسالة {msg_idx}]: دور غير مدعوم ({role_val})! المدعوم: system, user, assistant")
                        wrong_format_count += 1
                        errors += 1

            except json.JSONDecodeError:
                print(f"🚨 [السطر {idx+1}]: خطأ في الـ Syntax! ليس ملف JSON صالح.")
                wrong_format_count += 1
                errors += 1
                
    # النتيجة النهائية للفحص
    if errors == 0:
        print(f"✅ فحص ناجح 100%! الملف سليم تماماً ومتوافق مع Qwen 3.5 و Unsloth.")
        print(f"📊 إجمالي الأسطر المفحوصة والسليمة: {total_rows}")
        return True
    else:
        print(f"\n❌ تم العثور على أخطاء بحاجة لإصلاح:")
        print(f"   - عدد القيم الفارغة أو الـ null: {null_count}")
        print(f"   - عدد التنسيقات الخاطئة (Wrong Format): {wrong_format_count}")
        print(f"   - إجمالي الأخطاء الكلي: {errors} في أصل {total_rows} سطر.")
        return False

# تشغيل الفحص على الملفين
train_ok = validate_qwen_dataset(train_path)
eval_ok = validate_qwen_dataset(eval_path)

if train_ok and eval_ok:
    print("\n🎉 الداتا نظيفة وجاهزة للتدريب 100% من دون أي ريسك! فيك تبلش بـ Unsloth وأنت مطّمن.")
else:
    print("\n⚠️ انتبه! يرجى مراجعة الأخطاء المطبوعة فوق وتصحيحها قبل تشغيل الـ Trainer.")



--- ⏳ جاري الفحص الدقيق لملف: train.jsonl ---
✅ فحص ناجح 100%! الملف سليم تماماً ومتوافق مع Qwen 3.5 و Unsloth.
📊 إجمالي الأسطر المفحوصة والسليمة: 13499

--- ⏳ جاري الفحص الدقيق لملف: eval.jsonl ---
✅ فحص ناجح 100%! الملف سليم تماماً ومتوافق مع Qwen 3.5 و Unsloth.
📊 إجمالي الأسطر المفحوصة والسليمة: 1500

🎉 الداتا نظيفة وجاهزة للتدريب 100% من دون أي ريسك! فيك تبلش بـ Unsloth وأنت مطّمن.


In [13]:
import os

train_path = r"C:\Users\user\Desktop\Lebanese Assistant\lebanese-assistant\datasets\processed\train.jsonl"
temp_path = train_path + ".tmp"

print("⏳ جاري تنظيف وتصحيح ملف train.jsonl تلقائياً...")

cleaned_count = 0
with open(train_path, "r", encoding="utf-8") as f_in:
    with open(temp_path, "w", encoding="utf-8") as f_out:
        for idx, line in enumerate(f_in):
            # السطر 7789 بالـ Index ببلش من 7788 (لأن الـ Loop بتبلش من 0)
            # بس للاحتياط، السكربت رح يتخطى السطر اللي فيه المشكلة بالظبط
            if idx == 7788:
                print(f"✅ تم حذف السطر الفاسد رقم {idx+1} بنجاح.")
                continue
            f_out.write(line)
            cleaned_count += 1

# استبدال الملف القديم بالملف النظيف الجديد
os.remove(train_path)
os.rename(temp_path, train_path)

print(f"🎉 انتهى التصحيح! الملف الآن سليم تماماً وفيه {cleaned_count} سطر جاهز.")


⏳ جاري تنظيف وتصحيح ملف train.jsonl تلقائياً...
✅ تم حذف السطر الفاسد رقم 7789 بنجاح.
🎉 انتهى التصحيح! الملف الآن سليم تماماً وفيه 13499 سطر جاهز.


In [15]:
import json
import random
import os

train_path = r"C:\Users\user\Desktop\Lebanese Assistant\lebanese-assistant\datasets\processed\train.jsonl"

print("--- ⚙️ محاكاة فحص التنسيق لـ Qwen 3.5 و Unsloth ---")

if not os.path.exists(train_path):
    print("❌ خطأ: ملف train.jsonl غير موجود على المسار!")
else:
    # 1. قراءة عينات عشوائية من الملف
    samples = []
    with open(train_path, "r", encoding="utf-8") as f:
        lines = [line for line in f if line.strip()]
        # سحب 2 عينات عشوائية لنشوف شكلهم
        samples = random.sample(lines, min(2, len(lines)))
    
    # 2. محاكاة الـ Chat Template الافتراضي لـ Qwen 3.5 (ChatML)
    for idx, sample_line in enumerate(samples):
        data = json.loads(sample_line)
        messages = data["messages"]
        
        print(f"\n================ 👁️ عينة محاكاة رقم {idx+1} ================")
        print("📥 [الشكل بملف الـ JSONL]:")
        print(json.dumps(data, ensure_ascii=False, indent=2)[:400] + "...\n")
        
        print("🚀 [كيف سيري الموديل النص بفضل الـ ChatML الخاص بـ Qwen 3.5]:")
        
        # بناء يدوي يحاكي الـ Tokenizer الافتراضي لعائلة Qwen
        simulated_chat_template = ""
        for msg in messages:
            role = msg["role"]
            content = msg["content"]
            
            # قالب الـ ChatML الرسمي لـ Qwen 3.5 بيستخدم هول الـ Tokens بالظبط بالخلفية
            simulated_chat_template += f"<|im_start|>{role}\n{content}<|im_end|>\n"
            
        print(simulated_chat_template)
        print("=========================================================")

    print("\n✅ النتيجة التقنية:")
    print("1. حقل الـ 'messages' موجود كـ List وهيدا المطلوب لـ Unsloth.")
    print("2. الأدوار مقسمة لـ 'role' و 'content' داخل Dictionaries منفصلة.")
    print("3. الـ System Prompt موجود بأول كل سطر لتثبيت الهوية اللبنانية.")
    print("🔥 الداتا متطابقة 100% مع معايير Qwen 3.5 الافتراضية وجاهزة للـ Trainer فوراً!")


--- ⚙️ محاكاة فحص التنسيق لـ Qwen 3.5 و Unsloth ---

================ 👁️ عينة محاكاة رقم 1 ================
📥 [الشكل بملف الـ JSONL]:
{
  "messages": [
    {
      "role": "system",
      "content": "أنت مساعد ذكي لبناني. بتفهم وبتتكلم بالعامية اللبنانية وباللغة الإنجليزية بطلاقة، وأسلوبك دايماً قريب للقلب ومساعد."
    },
    {
      "role": "user",
      "content": "Have you used aws-vault for securely managing project credentials? If so, how does it compare to other methods mentioned in the text? Answer according to: I've been...

🚀 [كيف سيري الموديل النص بفضل الـ ChatML الخاص بـ Qwen 3.5]:
<|im_start|>system
أنت مساعد ذكي لبناني. بتفهم وبتتكلم بالعامية اللبنانية وباللغة الإنجليزية بطلاقة، وأسلوبك دايماً قريب للقلب ومساعد.<|im_end|>
<|im_start|>user
Have you used aws-vault for securely managing project credentials? If so, how does it compare to other methods mentioned in the text? Answer according to: I've been trying to get a manageable system for dealing with projects that need a l